# Benchmarks: M4, M3 e Rdatasets

Os dados oficiais são baixados **uma vez**, quando você pede, para o cache local (`~/.cache/autohybts/datasets` ou a variável `AUTOHYBTS_DATA_DIR`), com checksum SHA-256 e manifesto de origem e licença. Nada é baixado na instalação ou no `import`; com `offline=True`, só o cache é usado.

O teste oficial de cada série entra no experimento como **teste selado**: a busca e a seleção nunca o veem.

Isto **não** reproduz o score oficial das competições: é a avaliação de séries individuais.

In [ ]:
import autohybts

print("AutoHybTS", autohybts.__version__)

## 1. Catálogo

In [ ]:
from autohybts.datasets import list_datasets, load_benchmark

list_datasets()

In [ ]:
list_datasets(provider="m4", subset="daily").head()

## 2. Uma série do M4 com o teste oficial selado

`series.full()` junta treino e teste oficial; `test_size=len(series.test)` sela exatamente o teste oficial.

In [ ]:
series = load_benchmark("m4", subset="daily", series_id="D1")
series, series.descriptor.horizon, series.descriptor.license

In [ ]:
from autohybts import AutoHybridExperiment

exp = AutoHybridExperiment(
    fh=series.descriptor.horizon,
    test_size=len(series.test),
    seasonal_period=7,
    session_id=42,
    base_models=["naive", "seasonal_naive", "linear", "knn"],
    correctors=["linear"],
).fit(series.full())
result = exp.compare_models()
result.decisions[["family", "champion_before", "challenger", "accepted", "median_gain", "win_rate"]]

## 3. Avaliação no teste oficial (uma vez)

In [ ]:
final = exp.finalize_model(result.best)
exp.evaluate_external(final).metrics

## 4. M3

O M3 vem do pacote R `Mcomp` (CRAN). Ler o arquivo R requer o extra `datasets` (`pip install "autohybts[datasets]"`). Na primeira vez a conversão leva alguns segundos; depois o cache em CSV é usado.

In [ ]:
m3 = load_benchmark("m3", subset="monthly", series_id="N1402")
m3

Um painel com as três séries mensais mais longas do M3 (muitas têm só 48 pontos), cada uma com seu teste oficial selado:

In [ ]:
monthly = load_benchmark("m3", subset="monthly")
ids = list(monthly.metadata.sort_values("n_train").index[-3:])
panel = monthly.panel(ids)
exp_m3 = AutoHybridExperiment(
    fh=monthly.descriptor.horizon,
    test_size=monthly.descriptor.horizon,
    fold=1,
    budget="smoke",
    base_models=["naive", "seasonal_naive", "linear"],
    correctors=["linear"],
).fit(panel)
result_m3 = exp_m3.compare_models()
exp_m3.evaluate_external(exp_m3.finalize_model(result_m3.best)).metrics

## 5. Rdatasets

O Rdatasets é um catálogo heterogêneo: tempo, alvo e frequência são escolhidos explicitamente. `raw_dataset` mostra a tabela como publicada.

In [ ]:
from autohybts.datasets import load_dataset, raw_dataset

raw_dataset("rdatasets", package="datasets", name="AirPassengers").head()

In [ ]:
air = load_dataset(
    "rdatasets",
    package="datasets",
    name="AirPassengers",
    target="value",
    time_col="time",
    frequency="MS",
)
air.train.tail()